# 02. HIN 네트워크 EDA

이기종 그래프(HIN) 엣지리스트 기반 탐색적 데이터 분석  
`final_edgelist_with_trend.parquet` × `product_master_dataset.parquet`  

| 섹션 | 내용 |
|------|------|
| 0 | 환경 설정 |
| 1 | 노드 맵핑 & 타입 분류 |
| 2 | 기본 통계 |
| 3 | 차수(Degree) 분포 |
| 4 | 엣지 가중치 분포 |
| 5 | IP 노드 분석 |
| 6 | 속성 커버리지 |
| 7 | 제품-제품 동시구매 |
| 8 | NetworkX 그래프 구성 |
| 9 | 중심성 분석 |
| 10 | IP 서브그래프 시각화 |
| 11 | MBA 연관규칙 |
| 12 | 연관규칙 네트워크 시각화 |
| 13 | 200개 샘플 HIN 네트워크 |
| 14 | 두 상품 비교 네트워크 |


## 0. 환경 설정


In [ ]:
import os, sys, json, math
import pandas as pd
import polars as pl
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROC     = os.path.join(BASE_DIR, 'data', 'processed')
EDA_DIR  = os.path.join(BASE_DIR, 'eda')
OUT_DIR  = os.path.join(EDA_DIR, 'network_html')
os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print(f'BASE_DIR : {BASE_DIR}')
print(f'PROC     : {PROC}')
print(f'OUT_DIR  : {OUT_DIR}')


In [ ]:
prod      = pl.read_parquet(os.path.join(PROC, 'product_master_dataset.parquet'))
edges_all = pl.read_parquet(os.path.join(PROC, 'final_edgelist_with_trend.parquet'))

print(f'product_master : {prod.shape}')
print(f'  cols: {prod.columns}')
print(f'edges_all      : {edges_all.shape}')
print(f'  cols: {edges_all.columns}')
print()
print('엣지 타입별 개수:')
print(edges_all.group_by('edge_type').len().sort('len', descending=True))


In [ ]:
# 상품 코드-이름 매핑
code_to_name = dict(zip(prod['ITEM_CD'].to_list(), prod['ITEM_NM'].to_list()))
all_prod_codes = set(prod['ITEM_CD'].to_list())
print(f'전체 상품 수: {len(all_prod_codes):,}개')
print()
print('[product_master 샘플]')
print(prod.head(5))


## 1. 노드 맵핑 & 타입 분류
엣지 타입별로 분리하여 상품/속성/IP 노드 집합을 구성합니다.


In [ ]:
# ── 엣지 타입별 분리 (실제 스키마 기준) ──────────────────────────
# '제품-속성': trend_flag=0 일반 / trend_flag=1 트렌드
# 'IP-속성', 'IP-제품'
# '제품-제품' 동시구매 엣지는 현재 edgelist 미포함 (B2 POS 별도 처리 필요)
pa_edges = edges_all.filter(
    (pl.col('edge_type') == '제품-속성') & (pl.col('trend_flag') == 0)
).to_pandas()  # 일반 속성
tr_edges = edges_all.filter(
    (pl.col('edge_type') == '제품-속성') & (pl.col('trend_flag') == 1)
).to_pandas()  # 트렌드 속성
ip_attr  = edges_all.filter(pl.col('edge_type') == 'IP-속성').to_pandas()
ip_prod  = edges_all.filter(pl.col('edge_type') == 'IP-제품').to_pandas()
pp_edges = pd.DataFrame(columns=['src_node','dst_node','weight'])  # 빈 placeholder

# ── 노드 집합 ─────────────────────────────────────────────
product_nodes = (set(pa_edges['src_node'].tolist())
               | set(tr_edges['src_node'].tolist())
               | set(ip_prod['dst_node'].tolist()))
attr_nodes    = (set(pa_edges['dst_node'].tolist())
               | set(tr_edges['dst_node'].tolist())
               | set(ip_attr['dst_node'].tolist()))
ip_nodes      = (set(ip_attr['src_node'].tolist())
               | set(ip_prod['src_node'].tolist()))

print(f'상품 노드  : {len(product_nodes):,}개')
print(f'속성 노드  : {len(attr_nodes):,}개')
print(f'IP 노드    : {len(ip_nodes):,}개')
print(f'전체 노드  : {len(product_nodes | attr_nodes | ip_nodes):,}개')
print(f'일반속성 엣지  : {len(pa_edges):,}개')
print(f'트렌드속성 엣지: {len(tr_edges):,}개')
print(f'IP-속성 엣지   : {len(ip_attr):,}개')
print(f'IP-제품 엣지   : {len(ip_prod):,}개')


## 2. 기본 통계


In [ ]:
edge_type_cnt = edges_all.group_by('edge_type').len().sort('len', descending=True)
print('=== 엣지 타입별 개수 ===')
print(edge_type_cnt)

print()
if 'weight' in edges_all.columns:
    print('=== weight 기술통계 ===')
    print(edges_all.select(['weight']).describe())


## 3. 차수(Degree) 분포
상품당 보유 속성 수 / 속성당 연결 상품 수 분포를 확인합니다.


In [ ]:
# 상품당 연결 속성 수
prod_attr_deg = (pa_edges.groupby('src_node')['dst_node']
                .nunique().reset_index()
                .rename(columns={'dst_node': '속성수'}))
print('상품당 속성 수 통계:')
print(prod_attr_deg['속성수'].describe())

attr_prod_deg = (pa_edges.groupby('dst_node')['src_node']
                .nunique().reset_index()
                .rename(columns={'src_node': '연결상품수'}))
print()
print('속성당 연결 상품 수 통계:')
print(attr_prod_deg['연결상품수'].describe())

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(prod_attr_deg['속성수'], bins=40, color='#3B82F6', alpha=0.8, edgecolor='white')
axes[0].set_title('상품당 속성 수 분포'); axes[0].set_xlabel('속성 수'); axes[0].set_ylabel('상품 수')
axes[1].hist(attr_prod_deg['연결상품수'], bins=40, color='#10B981', alpha=0.8, edgecolor='white')
axes[1].set_title('속성당 연결 상품 수 분포'); axes[1].set_xlabel('연결 상품 수'); axes[1].set_ylabel('속성 수')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'degree_dist.png'), dpi=120, bbox_inches='tight')
plt.show()
print('✅ degree_dist.png 저장')


## 4. 엣지 가중치 분포
제품-제품 동시구매 엣지의 가중치 분포입니다.


In [ ]:
if 'weight' in pp_edges.columns:
    pp_w = pp_edges['weight'].astype(float)
    print('제품-제품 엣지 weight 통계:')
    print(pp_w.describe())
    clip95 = pp_w.quantile(0.95)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(pp_w.clip(upper=clip95), bins=50, color='#F97316', alpha=0.8, edgecolor='white')
    ax.set_title('제품-제품 엣지 가중치 분포 (95th percentile clip)')
    ax.set_xlabel('weight'); ax.set_ylabel('엣지 수')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'weight_dist.png'), dpi=120, bbox_inches='tight')
    plt.show()
    print('✅ weight_dist.png 저장')
else:
    print('weight 컬럼 없음')


## 5. IP(트렌드) 노드 분석
IP-속성, IP-제품 엣지를 통해 트렌드 IP 노드의 연결 패턴을 살펴봅니다.


In [ ]:
print(f'IP 노드 수: {len(ip_nodes)}개')
print(f'IP-속성 엣지: {len(ip_attr):,}개')
print(f'IP-제품 엣지: {len(ip_prod):,}개')

if len(ip_attr) > 0:
    ip_attr_cnt = (ip_attr.groupby('src_node')['dst_node']
                   .nunique().reset_index()
                   .rename(columns={'dst_node': '연결속성수'})
                   .sort_values('연결속성수', ascending=False))
    print()
    print('IP별 연결 속성 수 TOP 10:')
    print(ip_attr_cnt.head(10).to_string(index=False))

if len(ip_prod) > 0:
    ip_prod_cnt = (ip_prod.groupby('src_node')['dst_node']
                   .nunique().reset_index()
                   .rename(columns={'dst_node': '연결상품수'})
                   .sort_values('연결상품수', ascending=False))
    print()
    print('IP별 연결 상품 수 TOP 10:')
    print(ip_prod_cnt.head(10).to_string(index=False))


## 6. 속성 커버리지 분석
속성별 연결 상품 수(속성의 보편성)를 파악합니다.


In [ ]:
attr_prod_cnt = (pa_edges.groupby('dst_node')['src_node']
                 .nunique().reset_index()
                 .rename(columns={'src_node': '연결상품수'})
                 .sort_values('연결상품수', ascending=False))
print(f'전체 속성 수: {len(attr_prod_cnt):,}개')
print()
print('연결 상품 수 기준 상위 30개 속성:')
print(attr_prod_cnt.head(30).to_string(index=False))


## 7. 제품-제품 동시구매 분석
동시구매 가중치 상위 상품 쌍을 확인합니다.


In [ ]:
pp_sorted = pp_edges.sort_values('weight', ascending=False) if 'weight' in pp_edges.columns else pp_edges
print(f'제품-제품 엣지 총: {len(pp_sorted):,}개')
print()
print('동시구매 상위 20쌍:')
top20 = pp_sorted.head(20).copy()
top20['상품A명'] = top20['src_node'].map(code_to_name)
top20['상품B명'] = top20['dst_node'].map(code_to_name)
cols = ['상품A명', '상품B명', 'weight'] if 'weight' in top20.columns else ['상품A명', '상품B명']
print(top20[cols].to_string(index=False))


## 8. NetworkX 그래프 구성 (전체)
전체 HIN을 NetworkX `Graph`로 빌드하고 연결 컴포넌트를 확인합니다.


In [ ]:
G_full = nx.Graph()

for c in product_nodes:
    G_full.add_node(c, node_type='product')
for a in attr_nodes:
    G_full.add_node(a, node_type='attr')
for ip in ip_nodes:
    G_full.add_node(ip, node_type='ip')

for _, row in pa_edges.iterrows():
    G_full.add_edge(row['src_node'], row['dst_node'], weight=1.0, edge_type='pa')
for _, row in pp_edges.iterrows():
    w = float(row['weight']) if 'weight' in row else 1.0
    G_full.add_edge(row['src_node'], row['dst_node'], weight=w, edge_type='pp')
for _, row in ip_attr.iterrows():
    G_full.add_edge(row['src_node'], row['dst_node'], weight=1.0, edge_type='ip_attr')
for _, row in ip_prod.iterrows():
    G_full.add_edge(row['src_node'], row['dst_node'], weight=1.0, edge_type='ip_prod')

print(f'전체 그래프 — 노드: {G_full.number_of_nodes():,}  엣지: {G_full.number_of_edges():,}')
cc = list(nx.connected_components(G_full))
print(f'연결 컴포넌트: {len(cc)}개  (최대 컴포넌트: {max(len(c) for c in cc):,}개 노드)')


## 9. 중심성 분석 (상품 노드)
Degree / Betweenness / Eigenvector 중심성 평균 → **허브 점수** 산출.
노드 크기 결정 기준으로 사용됩니다.


In [ ]:
G_sub = G_full.subgraph(list(product_nodes | attr_nodes)).copy()
print(f'서브그래프 (상품+속성) — 노드: {G_sub.number_of_nodes():,}  엣지: {G_sub.number_of_edges():,}')

print('중심성 계산 중 (수분 소요)...')
deg_cent = nx.degree_centrality(G_sub)
btw_cent = nx.betweenness_centrality(G_sub, normalized=True, weight='weight')
try:
    eig_cent = nx.eigenvector_centrality(G_sub, max_iter=300, weight='weight')
    print('  ✓ Degree / Betweenness / Eigenvector')
except nx.PowerIterationFailedConvergence:
    eig_cent = deg_cent.copy()
    print('  ⚠ Eigenvector 미수렴 → Degree로 대체')


def normalize_cent(d):
    vmin, vmax = min(d.values()), max(d.values())
    span = vmax - vmin + 1e-9
    return {k: (v - vmin) / span for k, v in d.items()}


deg_n = normalize_cent(deg_cent)
btw_n = normalize_cent(btw_cent)
eig_n = normalize_cent(eig_cent)
hub_score = {n: (deg_n[n] + btw_n[n] + eig_n[n]) / 3 for n in G_sub.nodes()}

prod_hub = sorted(
    [(n, s) for n, s in hub_score.items() if n in product_nodes],
    key=lambda x: -x[1])
print()
print('허브 점수 TOP 20 상품:')
for cod, sc in prod_hub[:20]:
    print(f'  {code_to_name.get(cod, cod):30s}  {sc:.4f}')


## 10. IP 서브그래프 시각화
IP 노드와 직접 연결된 속성/상품만 추출하여 시각화합니다.


In [ ]:
# IP 연결 노드 서브그래프
ip_connected = set()
for _, row in ip_prod.iterrows():
    ip_connected.add(row['dst_node'])
for _, row in ip_attr.iterrows():
    ip_connected.add(row['dst_node'])

G_ip_sub = G_full.subgraph(ip_nodes | ip_connected).copy()
print(f'IP 서브그래프 — 노드: {G_ip_sub.number_of_nodes()}  엣지: {G_ip_sub.number_of_edges()}')

if G_ip_sub.number_of_nodes() == 0:
    print('IP 서브그래프 노드 없음 — 건너뜀')
else:
    pos_ip = nx.spring_layout(G_ip_sub, seed=42, k=3.0)
    colors = []
    for n in G_ip_sub.nodes():
        if n in ip_nodes:        colors.append('#EF4444')
        elif n in attr_nodes:    colors.append('#10B981')
        else:                    colors.append('#3B82F6')

    fig, ax = plt.subplots(figsize=(14, 10))
    nx.draw_networkx(G_ip_sub, pos_ip, ax=ax,
                     node_color=colors, node_size=200,
                     font_size=7, font_family='Malgun Gothic',
                     edge_color='#cccccc', alpha=0.85, with_labels=False)
    patches = [mpatches.Patch(color='#EF4444', label='IP 노드'),
               mpatches.Patch(color='#10B981', label='속성 노드'),
               mpatches.Patch(color='#3B82F6', label='상품 노드')]
    ax.legend(handles=patches, loc='upper left')
    ax.set_title('IP 연결 서브그래프', fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'ip_subgraph.png'), dpi=120, bbox_inches='tight')
    plt.show()
    print('✅ ip_subgraph.png 저장')


## 11. MBA 연관규칙 계산

`B2_POS_SALE` × `B4_ITEM_DV_INFO` 카테고리 레벨 MBA.  
온라인: `B3_OLN_ODR_DLVR` 거래번호 기준 분리 / 오프라인: 나머지.  
`mlxtend` Apriori → `rules_on_cat`, `rules_off_cat` 생성 후 섹션 12에서 HTML 출력.


In [ ]:
import polars as pl
from mlxtend.frequent_patterns import apriori, association_rules as mlxtend_assoc_rules
from mlxtend.preprocessing import TransactionEncoder


def compute_cat_rules(session_list, min_support=0.02, min_lift=1.0):
    """카테고리 세션 리스트(list of list) → mlxtend 연관규칙 DataFrame."""
    if len(session_list) < 10:
        print("  세션 부족 → 빈 DataFrame 반환")
        return pd.DataFrame()
    te = TransactionEncoder()
    te_arr = te.fit(session_list).transform(session_list)
    df_hot = pd.DataFrame(te_arr, columns=te.columns_)
    freq = apriori(df_hot, min_support=min_support, use_colnames=True)
    if freq.empty:
        print("  빈번 항목집합 없음 → 빈 DataFrame 반환")
        return pd.DataFrame()
    rules = mlxtend_assoc_rules(freq, metric='lift', min_threshold=min_lift)
    return rules.sort_values('lift', ascending=False).reset_index(drop=True)


# ── 데이터 로드 ────────────────────────────────────────────────
b2_lz = pl.scan_parquet(os.path.join(PROC, 'B2_POS_SALE.parquet'))
b3    = pl.read_parquet(os.path.join(PROC, 'B3_OLN_ODR_DLVR.parquet'),
                        columns=['점포코드', '거래번호']).unique()
b4_lz = pl.scan_parquet(os.path.join(PROC, 'B4_ITEM_DV_INFO.parquet')).select(['상품코드', '중분류명'])

# ── B2 × B4 카테고리 세션 (거래별 중분류 집합) ────────────────
print("카테고리 세션 생성 중 (수분 소요)...")
b2_cat = (
    b2_lz
    .join(b4_lz, on='상품코드', how='inner')
    .group_by(['점포코드', '거래번호'])
    .agg(pl.col('중분류명').unique().alias('cats'))
    .filter(pl.col('cats').list.len() >= 2)
    .collect()
)
print(f"  다중 카테고리 세션: {len(b2_cat):,}건")

# ── 온라인 / 오프라인 분리 ────────────────────────────────────
b2_on_df  = b2_cat.join(b3, on=['점포코드', '거래번호'], how='inner')
b2_off_df = b2_cat.join(b3, on=['점포코드', '거래번호'], how='anti')
print(f"  온라인 세션: {len(b2_on_df):,}건  |  오프라인 세션: {len(b2_off_df):,}건")

b2_on_sessions  = b2_on_df['cats'].to_list()
b2_off_sessions = b2_off_df['cats'].to_list()

# ── 연관규칙 계산 ─────────────────────────────────────────────
print("\n온라인 연관규칙 계산 중...")
rules_on_cat  = compute_cat_rules(b2_on_sessions,  min_support=0.02)

print("오프라인 연관규칙 계산 중...")
rules_off_cat = compute_cat_rules(b2_off_sessions, min_support=0.02)

print(f"\n온라인 규칙: {len(rules_on_cat):,}개  |  오프라인 규칙: {len(rules_off_cat):,}개")
if not rules_on_cat.empty:
    print("\n[온라인 TOP 5]")
    print(rules_on_cat[['antecedents','consequents','lift','confidence']].head(5).to_string(index=False))
if not rules_off_cat.empty:
    print("\n[오프라인 TOP 5]")
    print(rules_off_cat[['antecedents','consequents','lift','confidence']].head(5).to_string(index=False))


## 12. 연관규칙 네트워크 시각화

온라인/오프라인 TOP 20 규칙을 NetworkX로 시각화.  
antecedents → consequents 방향 엣지, 두께 = lift.


In [ ]:
from pyvis.network import Network

def draw_network_html(rules, title, output_path, top_n=20):
    """연관규칙 DataFrame → Pyvis 인터랙티브 HTML 저장.
    antecedents, consequents, lift, confidence, support 컬럼 필요.
    """
    if rules is None or rules.empty:
        print(f'{title}: 규칙 없음')
        return
    top = rules.nlargest(top_n, 'lift').reset_index(drop=True)
    lifts = top['lift'].values
    l_min, l_max = lifts.min(), lifts.max()
    def norm_w(lift):
        return float((lift - l_min) / (l_max - l_min + 1e-9) * 7 + 1)

    net = Network(
        height='780px', width='100%', directed=True,
        bgcolor='#f8f8f8', notebook=False, cdn_resources='in_line',
    )
    net.set_options('''
    {
      "physics": {
        "solver": "forceAtlas2Based",
        "forceAtlas2Based": {"springLength": 160, "springConstant": 0.04, "damping": 0.9},
        "minVelocity": 0.5, "stabilization": {"iterations": 200}
      },
      "edges": {
        "smooth": {"type": "curvedCW", "roundness": 0.15},
        "color": {"color": "#aaaaaa", "highlight": "#ff6b35", "hover": "#ff6b35"}
      },
      "nodes": {"font": {"size": 13, "face": "Malgun Gothic, sans-serif"}},
      "interaction": {"hover": true, "tooltipDelay": 80, "navigationButtons": true}
    }
    ''')

    added_nodes = set()
    for _, row in top.iterrows():
        ant  = ', '.join(sorted(list(row['antecedents'])))
        con  = ', '.join(sorted(list(row['consequents'])))
        lift = float(row['lift'])
        conf = float(row.get('confidence', float('nan')))
        supp = float(row.get('support',    float('nan')))
        for node in [ant, con]:
            if node not in added_nodes:
                net.add_node(
                    node, label=node,
                    title=f'<span style="font-family:Malgun Gothic,sans-serif"><b>{node}</b></span>',
                    color={
                        'background': '#4C8BF5', 'border': '#2a5fd4',
                        'highlight': {'background': '#ff6b35', 'border': '#c94e1e'},
                        'hover':     {'background': '#6ba3ff', 'border': '#2a5fd4'},
                    },
                    size=22, borderWidth=2, shadow=True,
                    font={'size': 12, 'color': '#ffffff', 'face': 'Malgun Gothic, sans-serif'},
                )
                added_nodes.add(node)
        tooltip = (
            f'<span style="font-family:Malgun Gothic,sans-serif">'
            f'<b>{ant} → {con}</b><br>'
            f'Lift: <b>{lift:.3f}</b><br>Confidence: {conf:.3f}<br>Support: {supp:.4f}</span>'
        )
        net.add_edge(ant, con, title=tooltip, width=norm_w(lift),
                     arrows={'to': {'enabled': True, 'scaleFactor': 0.8}})

    html = net.generate_html()
    font_css = (
        '<meta charset="utf-8">\n<style>\n'
        '  body, .vis-tooltip {'
        'font-family:"Malgun Gothic","Apple SD Gothic Neo",sans-serif !important;}\n'
        '  .vis-tooltip {font-size:13px; padding:8px 10px; border-radius:6px;}\n'
        '</style>\n'
    )
    html = html.replace('<head>', '<head>\n' + font_css, 1)
    html = html.replace('<title>network</title>', f'<title>{title}</title>', 1)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html)
    print(f'[{title}]  규칙 {len(top)}개  →  {output_path}')


In [ ]:
out_html = os.path.join(EDA_DIR, 'network_html')
os.makedirs(out_html, exist_ok=True)

draw_network_html(
    rules_on_cat,
    title='온라인 구매 연관규칙',
    output_path=os.path.join(out_html, 'network_online.html'),
    top_n=20,
)
draw_network_html(
    rules_off_cat,
    title='오프라인 구매 연관규칙',
    output_path=os.path.join(out_html, 'network_offline.html'),
    top_n=20,
)
print('완료 — eda/network_html/ 폴더 확인')


## 13. 200개 샘플 HIN 네트워크

IDF 차별성 기반 상품 200개 선발.  
- **정적 PNG**: `network_html/sample_network_200.png`  
- **인터랙티브 HTML**: `network_html/sample_network_200.html`  
  - 호버 → 레이블 표시  
  - 클릭 → 1-hop 강조  
  - 노드 크기: 중심성 허브점수 (상품) / 연결상품수 (속성)  
  - 상품:속성 = 1:2  


In [ ]:
"""
200개 샘플 HIN 네트워크
출력 1: sample_network_200.png  — 대표 정적 이미지 (matplotlib)
출력 2: sample_network_200.html — 인터랙티브 (클릭 → 1-hop 강조, 호버 → 레이블)

색상 규칙:
  앵커 6개   → 클러스터 색상 (노랑/주황/빨강)
  나머지 상품 → 다크네이비 #0D1F2D
  속성 노드  → #10B981 그린

상품 선택 전략:
  속성 IDF 기반 차별성 점수 상위 200개 선발
  (희귀 속성을 많이 가진 상품 = 다른 상품과 차이가 많이 나는 상품)
"""
import os, json, math
import pandas as pd
import polars as pl
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── 경로 ──────────────────────────────────────────────────────
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROC     = os.path.join(BASE_DIR, 'data', 'processed')
EDA_DIR  = os.path.join(BASE_DIR, 'eda')
OUT_DIR  = os.path.join(EDA_DIR, 'network_html')
os.makedirs(OUT_DIR, exist_ok=True)

# ── 데이터 로드 ────────────────────────────────────────────────
prod       = pl.read_parquet(os.path.join(PROC, 'product_master_dataset.parquet'))
edges_all  = pl.read_parquet(os.path.join(PROC, 'final_edgelist_with_trend.parquet'))
cluster_df = pd.read_excel(os.path.join(EDA_DIR, '00_NPD_lifecycle_clusters.xlsx'))

cluster_map  = {str(int(r['상품코드'])).zfill(6): int(r['cluster'])
                for _, r in cluster_df.iterrows()}
code_to_name = dict(zip(prod['ITEM_CD'].to_list(), prod['ITEM_NM'].to_list()))

# ══════════════════════════════════════════════════════════════
# 색상 정의
# ══════════════════════════════════════════════════════════════
CLUSTER_COLOR = {0: '#FCE378', 1: '#FF8C42', 2: '#EF4444'}
PROD_DEFAULT  = '#0D1F2D'   # 앵커 외 상품 — 다크네이비
ATTR_COLOR    = '#10B981'   # 속성 — 그린

# ── 앵커 상품 6개 (클러스터 색상 부여) ───────────────────────
ANCHOR_CODES = [
    '127814',  # PB)쪼코쪼코초코우유500ml      클러스터 0
    '125843',  # 이스타)도쿠시마라면큰컵        클러스터 0
    '125040',  # PB)이봉원의봉새우볶음밥260g    클러스터 1
    '125394',  # KBO)끝내기홈런미트부리또130g   클러스터 1
    '125340',  # KBO)롯데자이언츠거인단팥빵100g 클러스터 2
    '123788',  # PB)직화불막창180g              클러스터 2
]
anchor_set = set(ANCHOR_CODES)


def prod_color(code):
    if code in anchor_set:
        cl = cluster_map.get(code)
        return CLUSTER_COLOR.get(cl, PROD_DEFAULT)
    return PROD_DEFAULT


# ── 상품 200개 선발: 속성 IDF 차별성 점수 기반 ───────────────
all_codes = prod['ITEM_CD'].to_list()
N_ALL     = len(all_codes)
N_PROD    = 200

pa_idf_base = (
    edges_all.filter(pl.col('edge_type') == '제품-일반속성')
             .to_pandas()
)
attr_doc_cnt = pa_idf_base.groupby('dst_node')['src_node'].nunique()
idf_map      = {attr: math.log(N_ALL / cnt) for attr, cnt in attr_doc_cnt.items()}

prod_attr_sets = pa_idf_base.groupby('src_node')['dst_node'].apply(set).to_dict()
all_code_set   = set(all_codes)
distinct_score = {
    code: sum(idf_map.get(a, 0) for a in attrs)
    for code, attrs in prod_attr_sets.items()
    if code in all_code_set
}

# 앵커 우선 포함, 나머지는 차별성 점수 내림차순
ranked = sorted(distinct_score.items(), key=lambda x: -x[1])
sample_200 = list(ANCHOR_CODES)
for code, score in ranked:
    if len(sample_200) >= N_PROD:
        break
    if code not in anchor_set:
        sample_200.append(code)
if len(sample_200) < N_PROD:
    print(f'⚠ 선발 상품 {len(sample_200)}개 (목표 {N_PROD}개에 미달)')
sample_set = set(sample_200)
print(f'차별성 기반 상품 선발 완료: {len(sample_200)}개')

# ── 엣지 필터링 ───────────────────────────────────────────────
pa_all = (
    edges_all.filter(pl.col('edge_type') == '제품-일반속성')
             .filter(pl.col('src_node').is_in(sample_set))
             .to_pandas()
)

# 속성 2:1 비율 (연결 상품 수 기준 상위 400개)
attr_degree = (pa_all.groupby('dst_node')['src_node']
               .nunique().reset_index()
               .rename(columns={'src_node': 'prod_count'})
               .sort_values('prod_count', ascending=False))
N_ATTR    = 2 * N_PROD
top_attrs = set(attr_degree.head(N_ATTR)['dst_node'].tolist())
pa_edges  = pa_all[pa_all['dst_node'].isin(top_attrs)]

pp_edges = (
    edges_all.filter(pl.col('edge_type') == '제품-제품')
             .filter(pl.col('src_node').is_in(sample_set) &
                     pl.col('dst_node').is_in(sample_set))
             .sort('weight', descending=True)
             .head(3000)
             .to_pandas()
)

print(f'상품: {len(sample_200)}개  속성: {len(top_attrs)}개  비율 {len(top_attrs)/len(sample_200):.1f}:1')
print(f'제품-속성 엣지: {len(pa_edges):,}  제품-제품 엣지: {len(pp_edges):,}')

# ══════════════════════════════════════════════════════════════
# NetworkX 그래프 + 중심성 계산
# ══════════════════════════════════════════════════════════════
G = nx.Graph()
for c in sample_200:
    G.add_node(c, node_type='product')
for a in top_attrs:
    G.add_node(a, node_type='attr')
for _, row in pa_edges.iterrows():
    G.add_edge(row['src_node'], row['dst_node'], weight=1.0)
for _, row in pp_edges.iterrows():
    G.add_edge(row['src_node'], row['dst_node'], weight=float(row['weight']))

print(f'NetworkX 그래프 — 노드: {G.number_of_nodes():,}  엣지: {G.number_of_edges():,}')
print('중심성 계산 중...')

deg_cent = nx.degree_centrality(G)
btw_cent = nx.betweenness_centrality(G, normalized=True, weight='weight')
try:
    eig_cent = nx.eigenvector_centrality(G, max_iter=500, weight='weight')
    print('  ✓ Degree / Betweenness / Eigenvector')
except nx.PowerIterationFailedConvergence:
    eig_cent = deg_cent.copy()
    print('  ✓ Degree / Betweenness  ⚠ Eigenvector→Degree 대체')


def normalize(d):
    vmin, vmax = min(d.values()), max(d.values())
    span = vmax - vmin + 1e-9
    return {k: (v - vmin) / span for k, v in d.items()}


deg_n = normalize(deg_cent)
btw_n = normalize(btw_cent)
eig_n = normalize(eig_cent)
hub_score = {n: (deg_n[n] + btw_n[n] + eig_n[n]) / 3 for n in G.nodes()}

# 노드 크기
PROD_SZ_MIN, PROD_SZ_MAX = 10, 45
ATTR_SZ_MIN, ATTR_SZ_MAX = 12, 60
attr_cnt_map = attr_degree.set_index('dst_node')['prod_count'].to_dict()
attr_max     = attr_degree['prod_count'].max()


def prod_size(code):
    return PROD_SZ_MIN + hub_score.get(code, 0) * (PROD_SZ_MAX - PROD_SZ_MIN)


def attr_size(attr):
    return ATTR_SZ_MIN + (attr_cnt_map.get(attr, 1) / attr_max) * (ATTR_SZ_MAX - ATTR_SZ_MIN)


# spring_layout (공유 좌표)
print('layout 계산 중...')
pos = nx.spring_layout(G, seed=42, k=2.5, iterations=80)
SCALE = 1500
coord = {n: (pos[n][0] * SCALE, pos[n][1] * SCALE) for n in G.nodes()}

# ══════════════════════════════════════════════════════════════
# [출력 1] 정적 PNG — matplotlib
# ══════════════════════════════════════════════════════════════
print('\nPNG 생성 중...')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(20, 16))
fig.patch.set_alpha(0)
ax.set_facecolor('none')

nx_pos = pos

edge_prod_attr = [(row['src_node'], row['dst_node']) for _, row in pa_edges.iterrows()]
edge_prod_prod = [(row['src_node'], row['dst_node']) for _, row in pp_edges.iterrows()]

nx.draw_networkx_edges(G, nx_pos, edgelist=edge_prod_attr,
                       edge_color='#c8d6e5', width=0.4, alpha=0.5, ax=ax)
nx.draw_networkx_edges(G, nx_pos, edgelist=edge_prod_prod,
                       edge_color='#aaaaaa', width=0.6, alpha=0.35, ax=ax)

non_anchor = [c for c in sample_200 if c not in anchor_set]
non_anchor_sizes = [prod_size(c) * 8 for c in non_anchor]
nx.draw_networkx_nodes(G, nx_pos, nodelist=non_anchor,
                       node_color=PROD_DEFAULT, node_size=non_anchor_sizes,
                       alpha=0.75, ax=ax)

attr_list  = list(top_attrs)
attr_sizes = [attr_size(a) * 10 for a in attr_list]
nx.draw_networkx_nodes(G, nx_pos, nodelist=attr_list,
                       node_color=ATTR_COLOR, node_size=attr_sizes,
                       alpha=0.8, ax=ax)

top_attrs_label = attr_degree.head(30)['dst_node'].tolist()
attr_label_map = {a: a for a in top_attrs_label if a in G.nodes()}
nx.draw_networkx_labels(G, nx_pos, labels=attr_label_map,
                        font_size=6, font_color='white',
                        font_family='Malgun Gothic', ax=ax)

for code in ANCHOR_CODES:
    cl  = cluster_map.get(code)
    col = CLUSTER_COLOR.get(cl, PROD_DEFAULT)
    sz  = prod_size(code) * 25
    nx.draw_networkx_nodes(G, nx_pos, nodelist=[code],
                           node_color=col, node_size=sz,
                           edgecolors='white', linewidths=2.5, ax=ax)
    nm = code_to_name.get(code, code)
    nx.draw_networkx_labels(G, nx_pos, labels={code: nm},
                            font_size=7, font_color='black',
                            font_weight='bold', font_family='Malgun Gothic', ax=ax)

legend_patches = [
    mpatches.Patch(color='#FCE378', label='클러스터 0 (균일수요)'),
    mpatches.Patch(color='#FF8C42', label='클러스터 1'),
    mpatches.Patch(color='#EF4444', label='클러스터 2 (단기흥행)'),
    mpatches.Patch(color='#0D1F2D', label='기타 상품'),
    mpatches.Patch(color='#10B981', label='속성 키워드'),
]
ax.legend(handles=legend_patches, loc='upper left', fontsize=11,
          framealpha=0.9, edgecolor='#e2e8f0')
ax.set_title('7-Eleven NPD HIN 네트워크 — 200개 상품 샘플\n(노드 크기: 중심성 허브점수 / 상품:속성 = 1:2)',
             fontsize=15, fontweight='bold', pad=15)
ax.axis('off')
plt.tight_layout()

png_path = os.path.join(OUT_DIR, 'sample_network_200.png')
plt.savefig(png_path, dpi=180, bbox_inches='tight', transparent=True)
plt.close()
print(f'✅ PNG: {png_path}')

# ══════════════════════════════════════════════════════════════
# [출력 2] 인터랙티브 HTML
# ══════════════════════════════════════════════════════════════
print('HTML 생성 중...')

vis_nodes = []
vis_edges = []

for code in sample_200:
    bg        = prod_color(code)
    is_anchor = code in anchor_set
    name      = code_to_name.get(code, code)
    cl        = cluster_map.get(code)
    cl_label  = f'클러스터 {cl}' if cl is not None else '비NPD'
    x, y      = coord.get(code, (0, 0))

    vis_nodes.append({
        'id':          code,
        'nodeType':    'product',
        'label':       '',
        'hiddenLabel': name[:10] + '…' if len(name) > 10 else name,
        'shape':       'dot',
        'size':        round(prod_size(code), 1),
        'color': {
            'background': bg,
            'border':     '#ffffff' if is_anchor else bg,
            'highlight':  {'background': bg, 'border': '#ffffff'},
            'hover':      {'background': bg, 'border': '#ffffff'},
        },
        'font':        {'color': '#f8fafc', 'size': 10,
                        'face': 'Malgun Gothic, Apple SD Gothic Neo, sans-serif'},
        'borderWidth': 2.5 if is_anchor else 1,
        'shadow':      is_anchor,
        'physics':     False,
        'x':           round(x, 2),
        'y':           round(y, 2),
        'title': (
            f'<span style="font-family:\'Malgun Gothic\',sans-serif;">'
            f'🛍️ <b>{name}</b><br>코드: {code}<br>{cl_label}'
            + (' ★앵커' if is_anchor else '') +
            f'<br>허브점수: {hub_score.get(code,0):.3f}'
            f'</span>'
        ),
    })

for attr in top_attrs:
    x, y = coord.get(attr, (0, 0))
    cnt  = attr_cnt_map.get(attr, 1)
    vis_nodes.append({
        'id':          attr,
        'nodeType':    'attr',
        'label':       '',
        'hiddenLabel': attr,
        'shape':       'dot',
        'size':        round(attr_size(attr), 1),
        'color': {
            'background': ATTR_COLOR,
            'border':     '#059669',
            'highlight':  {'background': '#34D399', 'border': '#ffffff'},
            'hover':      {'background': '#34D399', 'border': '#ffffff'},
        },
        'font':        {'color': '#f0f4f8', 'size': 10,
                        'face': 'Malgun Gothic, Apple SD Gothic Neo, sans-serif'},
        'borderWidth': 1,
        'shadow':      True,
        'physics':     False,
        'x':           round(x, 2),
        'y':           round(y, 2),
        'title': (
            f'<span style="font-family:\'Malgun Gothic\',sans-serif;">'
            f'🔑 속성: <b>{attr}</b><br>연결 상품 수: {cnt}개</span>'
        ),
    })

eid = 0
for _, row in pa_edges.iterrows():
    vis_edges.append({'id': eid, 'from': row['src_node'], 'to': row['dst_node'],
                      'color': {'color': '#c8d6e5'}, 'width': 1})
    eid += 1

if len(pp_edges) > 0:
    w_min, w_max = pp_edges['weight'].min(), pp_edges['weight'].max()
    for _, row in pp_edges.iterrows():
        w = (row['weight'] - w_min) / (w_max - w_min + 1e-9) * 3 + 0.5
        vis_edges.append({'id': eid, 'from': row['src_node'], 'to': row['dst_node'],
                          'color': {'color': '#aaaaaa'}, 'width': round(float(w), 2)})
        eid += 1

nodes_json = json.dumps(vis_nodes, ensure_ascii=False)
edges_json = json.dumps(vis_edges, ensure_ascii=False)

js_highlight = f"""
<script>
var activeNode = null;
var origNodeColor = {{}};
var origEdgeColor = {{}};
var origEdgeWidth = {{}};

nodes.forEach(function(n) {{
  origNodeColor[n.id] = JSON.parse(JSON.stringify(n.color));
}});
edges.forEach(function(e) {{
  origEdgeColor[e.id] = JSON.parse(JSON.stringify(e.color));
  origEdgeWidth[e.id] = e.width || 1;
}});

network.on('hoverNode', function(params) {{
  var n = nodes.get(params.node);
  if (n && n.hiddenLabel) {{
    nodes.update({{id: params.node, label: n.hiddenLabel}});
  }}
}});

network.on('blurNode', function(params) {{
  nodes.update({{id: params.node, label: ''}});
}});

network.on('click', function(params) {{
  if (params.nodes.length === 0) {{
    restoreAll();
    activeNode = null;
    return;
  }}
  var nid = params.nodes[0];
  if (activeNode === nid) {{
    restoreAll();
    activeNode = null;
    return;
  }}
  activeNode = nid;
  applyHighlight(nid);
}});

function applyHighlight(nodeId) {{
  var neighbors = new Set(network.getConnectedNodes(nodeId));
  neighbors.add(nodeId);
  var connectedEdges = new Set(network.getConnectedEdges(nodeId));

  var nodeUpd = [];
  nodes.forEach(function(n) {{
    if (neighbors.has(n.id)) {{
      nodeUpd.push({{id: n.id, color: JSON.parse(JSON.stringify(origNodeColor[n.id])), opacity: 1.0, borderWidth: 3}});
    }} else {{
      nodeUpd.push({{
        id: n.id,
        color: {{background:'#e8ecef', border:'#ced4da',
                highlight:{{background:'#e8ecef',border:'#ced4da'}},
                hover:{{background:'#e8ecef',border:'#ced4da'}}}},
        opacity: 0.2
      }});
    }}
  }});
  nodes.update(nodeUpd);

  var edgeUpd = [];
  edges.forEach(function(e) {{
    if (connectedEdges.has(e.id)) {{
      edgeUpd.push({{id: e.id, color:{{color:'#F97316'}}, width: 2.5}});
    }} else {{
      edgeUpd.push({{id: e.id, color:{{color:'#eeeeee'}}, width: 0.3}});
    }}
  }});
  edges.update(edgeUpd);
}}

function restoreAll() {{
  var nodeUpd = [];
  nodes.forEach(function(n) {{
    nodeUpd.push({{id: n.id, color: JSON.parse(JSON.stringify(origNodeColor[n.id])), opacity: 1.0}});
  }});
  nodes.update(nodeUpd);
  var edgeUpd = [];
  edges.forEach(function(e) {{
    edgeUpd.push({{id: e.id, color: JSON.parse(JSON.stringify(origEdgeColor[e.id])),
                  width: origEdgeWidth[e.id]}});
  }});
  edges.update(edgeUpd);
}}
</script>
"""

html = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>200개 샘플 HIN — 중심성 허브 · 클릭 1-hop 강조</title>
<style>
  body {{ margin: 0; font-family: 'Malgun Gothic', 'Apple SD Gothic Neo', 'Nanum Gothic', sans-serif; background: #ffffff; }}
  #network {{ width: 100%; height: 100vh; border: none; }}
  .vis-tooltip {{
    font-family: 'Malgun Gothic', sans-serif !important;
    font-size: 13px; padding: 8px 12px; border-radius: 6px;
    max-width: 320px; background: rgba(255,255,255,0.97);
    border: 1px solid #e2e8f0; color: #1e293b;
    box-shadow: 0 4px 12px rgba(0,0,0,0.15);
  }}
</style>
<script src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
</head>
<body>
<div id="network"></div>
<script>
  var nodesRaw = {nodes_json};
  nodesRaw = nodesRaw.map(function(n) {{
    if (n.title) {{
      var div = document.createElement('div');
      div.style.cssText = 'font-family:Malgun Gothic,sans-serif;font-size:13px;line-height:1.6';
      div.innerHTML = n.title;
      n.title = div;
    }}
    return n;
  }});
  var nodes = new vis.DataSet(nodesRaw);
  var edges = new vis.DataSet({edges_json});
  var container = document.getElementById('network');
  var network = new vis.Network(container,
    {{nodes: nodes, edges: edges}},
    {{
      nodes:   {{borderWidth: 2, shadow: true}},
      edges:   {{color: {{color: "#aaaaaa"}}, smooth: {{type: "continuous"}}}},
      physics: {{enabled: false}},
      interaction: {{
        dragNodes: true, dragView: true, zoomView: true,
        hover: true, tooltipDelay: 80,
        navigationButtons: true, keyboard: true
      }}
    }}
  );
</script>
{js_highlight}
</body>
</html>"""

html_path = os.path.join(OUT_DIR, 'sample_network_200.html')
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(html)
print(f'✅ HTML: {html_path}')
print(f'\n노드 {len(vis_nodes)}개  엣지 {len(vis_edges)}개')
print(f'  앵커 6개 (클러스터 색상) + 비앵커 {len(sample_200)-6}개 (다크네이비) + 속성 {len(top_attrs)}개')


## 14. 두 상품 비교 네트워크

KBO 앵커 두 상품의 속성/동시구매 패턴 비교.  
- **A**: KBO)끝내기홈런미트부리또130g (`#3B82F6`)  
- **B**: KBO)롯데자이언츠거인단팥빵100g (`#F97316`)  
- 공유 속성 (보라 `#7C3AED`) / A 전용 (하늘 `#93C5FD`) / B 전용 (주황 `#FDBA74`)  
- 동시구매 파트너 상위 8개씩 외곽 호(arc) 배치  
- **출력**: `network_html/compare_two_products.html`  


In [ ]:
"""
두 상품 비교 HIN 네트워크
A: KBO)끝내기홈런미트부리또130g  (#3B82F6, x=-700)
B: KBO)롯데자이언츠거인단팥빵100g (#F97316, x=+700)

레이아웃:
  A 노드 — 왼쪽 고정
  B 노드 — 오른쪽 고정
  A 전용 속성 (#93C5FD) — 왼쪽 반원
  B 전용 속성 (#FDBA74) — 오른쪽 반원
  공유 속성  (#7C3AED) — 가운데 세로 배열
  동시구매 파트너 (상위 8개씩) — 외곽 호(arc)

출력: network_html/compare_two_products.html
"""
import os, json, math
import pandas as pd
import polars as pl
import networkx as nx

# ── 경로 ──────────────────────────────────────────────────────
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROC     = os.path.join(BASE_DIR, 'data', 'processed')
EDA_DIR  = os.path.join(BASE_DIR, 'eda')
OUT_DIR  = os.path.join(EDA_DIR, 'network_html')
os.makedirs(OUT_DIR, exist_ok=True)

# ── 두 앵커 상품 ──────────────────────────────────────────────
CODE_A = '125394'   # KBO)끝내기홈런미트부리또130g
CODE_B = '125340'   # KBO)롯데자이언츠거인단팥빵100g
COLOR_A = '#3B82F6'
COLOR_B = '#F97316'
COLOR_A_ONLY  = '#93C5FD'   # A 전용 속성 (하늘)
COLOR_B_ONLY  = '#FDBA74'   # B 전용 속성 (주황)
COLOR_SHARED  = '#7C3AED'   # 공유 속성 (보라)
COLOR_PARTNER = '#64748B'   # 동시구매 파트너 (회색)

# ── 데이터 로드 ────────────────────────────────────────────────
prod      = pl.read_parquet(os.path.join(PROC, 'product_master_dataset.parquet'))
edges_all = pl.read_parquet(os.path.join(PROC, 'final_edgelist_with_trend.parquet'))

code_to_name = dict(zip(prod['ITEM_CD'].to_list(), prod['ITEM_NM'].to_list()))
NAME_A = code_to_name.get(CODE_A, CODE_A)
NAME_B = code_to_name.get(CODE_B, CODE_B)
print(f'A: {NAME_A}')
print(f'B: {NAME_B}')

# ── 속성 엣지 ─────────────────────────────────────────────────
pa = edges_all.filter(pl.col('edge_type') == '제품-일반속성').to_pandas()
attrs_A = set(pa[pa['src_node'] == CODE_A]['dst_node'].tolist())
attrs_B = set(pa[pa['src_node'] == CODE_B]['dst_node'].tolist())
attrs_shared  = attrs_A & attrs_B
attrs_A_only  = attrs_A - attrs_B
attrs_B_only  = attrs_B - attrs_A

print(f'A 전용 속성: {len(attrs_A_only)}개  B 전용 속성: {len(attrs_B_only)}개  공유: {len(attrs_shared)}개')

# 속성 수 제한 (화면 가독성)
MAX_ATTR = 20
attrs_A_only  = set(list(attrs_A_only)[:MAX_ATTR])
attrs_B_only  = set(list(attrs_B_only)[:MAX_ATTR])
attrs_shared  = set(list(attrs_shared)[:MAX_ATTR])

# ── 동시구매 파트너 (상위 8개) ────────────────────────────────
pp = edges_all.filter(pl.col('edge_type') == '제품-제품').to_pandas()

def top_partners(code, n=8):
    mask = (pp['src_node'] == code) | (pp['dst_node'] == code)
    sub  = pp[mask].copy()
    sub['partner'] = sub.apply(
        lambda r: r['dst_node'] if r['src_node'] == code else r['src_node'], axis=1)
    sub = sub.sort_values('weight', ascending=False)
    return sub[['partner', 'weight']].head(n).values.tolist()

partners_A = top_partners(CODE_A)
partners_B = top_partners(CODE_B)

# 파트너 코드 집합 (양쪽 중복 제거)
partner_set = {p for p, _ in partners_A} | {p for p, _ in partners_B}
print(f'동시구매 파트너: A={len(partners_A)}개  B={len(partners_B)}개')

# ══════════════════════════════════════════════════════════════
# 좌표 계산 (fixed physics=false)
# ══════════════════════════════════════════════════════════════
CX_A, CX_B = -700, 700
CY_C = 0          # 가운데 Y 중심

def semicircle_coords(n, cx, side='left', r=280):
    """반원 좌표 — side='left': π/2 ~ 3π/2, side='right': -π/2 ~ π/2"""
    if n == 0:
        return []
    if side == 'left':
        angles = [math.pi/2 + i * math.pi / max(n - 1, 1) for i in range(n)]
    else:
        angles = [-math.pi/2 + i * math.pi / max(n - 1, 1) for i in range(n)]
    return [(round(cx + r * math.cos(a), 1), round(CY_C + r * math.sin(a), 1))
            for a in angles]

def vertical_coords(n, cx=0, y_span=500):
    if n == 0:
        return []
    step = y_span / max(n - 1, 1)
    return [(cx, round(-y_span / 2 + i * step, 1)) for i in range(n)]

def arc_coords(n, cx, side='left', r=550):
    """파트너 외곽 호 좌표"""
    if n == 0:
        return []
    if side == 'left':
        angles = [math.pi * 0.6 + i * (math.pi * 0.8) / max(n - 1, 1) for i in range(n)]
    else:
        angles = [-math.pi * 0.4 - i * (math.pi * 0.8) / max(n - 1, 1) for i in range(n)]
    return [(round(cx + r * math.cos(a), 1), round(CY_C + r * math.sin(a), 1))
            for a in angles]


# 좌표 매핑
coord_map = {}
coord_map[CODE_A] = (CX_A, 0)
coord_map[CODE_B] = (CX_B, 0)

a_only_list  = list(attrs_A_only)
b_only_list  = list(attrs_B_only)
shared_list  = list(attrs_shared)

for node, (x, y) in zip(a_only_list,  semicircle_coords(len(a_only_list),  CX_A, 'left')):
    coord_map[node] = (x, y)
for node, (x, y) in zip(b_only_list,  semicircle_coords(len(b_only_list),  CX_B, 'right')):
    coord_map[node] = (x, y)
for node, (x, y) in zip(shared_list,  vertical_coords(len(shared_list))):
    coord_map[node] = (x, y)

pa_list = [p for p, _ in partners_A]
pb_list = [p for p, _ in partners_B]
for node, (x, y) in zip(pa_list, arc_coords(len(pa_list), CX_A, 'left')):
    coord_map[node] = (x, y)
for node, (x, y) in zip(pb_list, arc_coords(len(pb_list), CX_B, 'right')):
    coord_map.setdefault(node, (x, y))   # B 파트너가 A 파트너와 겹칠 경우 우선 A좌표 유지

# ══════════════════════════════════════════════════════════════
# vis.js 노드 / 엣지 빌드
# ══════════════════════════════════════════════════════════════
vis_nodes = []
vis_edges = []
eid = 0


def push_node(nid, label, color, border, size, shape='dot', bold=False):
    x, y = coord_map.get(nid, (0, 0))
    name_full = code_to_name.get(nid, nid)
    vis_nodes.append({
        'id':          nid,
        'label':       '',
        'hiddenLabel': label,
        'shape':       shape,
        'size':        size,
        'color': {'background': color, 'border': border,
                  'highlight': {'background': color, 'border': '#ffffff'},
                  'hover':     {'background': color, 'border': '#ffffff'}},
        'font':        {'color': '#ffffff', 'size': 11, 'bold': bold,
                        'face': 'Malgun Gothic, sans-serif'},
        'borderWidth': 3 if bold else 1,
        'shadow':      bold,
        'physics':     False,
        'x':           x,
        'y':           y,
        'title':       f'<div style="font-family:Malgun Gothic,sans-serif;font-size:13px">'
                       f'<b>{name_full}</b><br>코드: {nid}</div>',
    })


def push_edge(src, dst, color, width=1.0):
    global eid
    vis_edges.append({'id': eid, 'from': src, 'to': dst,
                      'color': {'color': color}, 'width': width})
    eid += 1


# 앵커 상품 A, B
push_node(CODE_A, NAME_A[:12] + ('…' if len(NAME_A) > 12 else ''),
          COLOR_A, '#ffffff', 40, bold=True)
push_node(CODE_B, NAME_B[:12] + ('…' if len(NAME_B) > 12 else ''),
          COLOR_B, '#ffffff', 40, bold=True)

# 속성 노드
for attr in a_only_list:
    push_node(attr, attr, COLOR_A_ONLY, '#60A5FA', 14)
    push_edge(CODE_A, attr, COLOR_A_ONLY, 1.2)

for attr in b_only_list:
    push_node(attr, attr, COLOR_B_ONLY, '#FB923C', 14)
    push_edge(CODE_B, attr, COLOR_B_ONLY, 1.2)

for attr in shared_list:
    push_node(attr, attr, COLOR_SHARED, '#A78BFA', 16)
    push_edge(CODE_A, attr, COLOR_SHARED, 1.5)
    push_edge(CODE_B, attr, COLOR_SHARED, 1.5)

# 동시구매 파트너
added_partners = set()
w_vals = [w for _, w in partners_A + partners_B]
w_min, w_max = (min(w_vals), max(w_vals)) if w_vals else (1, 1)

for partner, w in partners_A:
    if partner not in added_partners and partner not in {CODE_A, CODE_B}:
        pname = code_to_name.get(partner, partner)
        push_node(partner, pname[:10] + ('…' if len(pname) > 10 else ''),
                  COLOR_PARTNER, '#94A3B8', 12)
        added_partners.add(partner)
    edge_w = 0.5 + (w - w_min) / (w_max - w_min + 1e-9) * 3
    push_edge(CODE_A, partner, '#93C5FD', round(edge_w, 2))

for partner, w in partners_B:
    if partner not in added_partners and partner not in {CODE_A, CODE_B}:
        pname = code_to_name.get(partner, partner)
        push_node(partner, pname[:10] + ('…' if len(pname) > 10 else ''),
                  COLOR_PARTNER, '#94A3B8', 12)
        added_partners.add(partner)
    edge_w = 0.5 + (w - w_min) / (w_max - w_min + 1e-9) * 3
    push_edge(CODE_B, partner, '#FDBA74', round(edge_w, 2))

nodes_json = json.dumps(vis_nodes, ensure_ascii=False)
edges_json = json.dumps(vis_edges, ensure_ascii=False)

# ══════════════════════════════════════════════════════════════
# HTML 출력
# ══════════════════════════════════════════════════════════════
html = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>두 상품 비교 네트워크</title>
<style>
  body {{ margin: 0; background: #ffffff;
         font-family: 'Malgun Gothic', 'Apple SD Gothic Neo', sans-serif; }}
  #network {{ width: 100%; height: 100vh; }}
  #legend {{
    position: fixed; top: 16px; left: 16px; z-index: 9999;
    background: rgba(255,255,255,0.97); border: 1px solid #e2e8f0;
    border-radius: 10px; padding: 14px 18px; font-size: 13px;
    color: #1e293b; line-height: 2; min-width: 210px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.1);
  }}
  .vis-tooltip {{
    font-family: 'Malgun Gothic', sans-serif !important;
    font-size: 13px; padding: 8px 12px; border-radius: 6px;
    max-width: 280px; background: rgba(255,255,255,0.97);
    border: 1px solid #e2e8f0; color: #1e293b;
    box-shadow: 0 4px 12px rgba(0,0,0,0.15);
  }}
</style>
<script src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
</head>
<body>
<div id="legend">
  <b style="font-size:14px">🔵 A 상품</b><br>
  <span style="color:{COLOR_A}">●</span> {NAME_A[:16]}<br>
  <b style="font-size:14px">🟠 B 상품</b><br>
  <span style="color:{COLOR_B}">●</span> {NAME_B[:16]}<br><br>
  <span style="color:{COLOR_A_ONLY}">●</span> A 전용 속성<br>
  <span style="color:{COLOR_B_ONLY}">●</span> B 전용 속성<br>
  <span style="color:{COLOR_SHARED}">●</span> 공유 속성<br>
  <span style="color:{COLOR_PARTNER}">●</span> 동시구매 파트너<br><br>
  <span style="font-size:11px;color:#64748b">
    클릭: 1-hop 강조 / 호버: 레이블<br>
    빈 공간 클릭: 초기화
  </span>
</div>
<div id="network"></div>
<script>
  var nodesRaw = {nodes_json};
  nodesRaw = nodesRaw.map(function(n) {{
    if (n.title) {{
      var div = document.createElement('div');
      div.style.cssText = 'font-family:Malgun Gothic,sans-serif;font-size:13px;line-height:1.6';
      div.innerHTML = n.title;
      n.title = div;
    }}
    return n;
  }});
  var nodes = new vis.DataSet(nodesRaw);
  var edges = new vis.DataSet({edges_json});
  var container = document.getElementById('network');
  var network = new vis.Network(container,
    {{nodes: nodes, edges: edges}},
    {{
      nodes:   {{borderWidth: 2, shadow: false}},
      edges:   {{smooth: {{type: 'continuous'}}}},
      physics: {{enabled: false}},
      interaction: {{
        dragNodes: true, dragView: true, zoomView: true,
        hover: true, tooltipDelay: 80,
        navigationButtons: true, keyboard: true
      }}
    }}
  );

  // 원본 색상 저장
  var origNodeColor = {{}};
  var origEdgeColor = {{}};
  var origEdgeWidth = {{}};
  nodes.forEach(function(n) {{ origNodeColor[n.id] = JSON.parse(JSON.stringify(n.color)); }});
  edges.forEach(function(e) {{ origEdgeColor[e.id] = JSON.parse(JSON.stringify(e.color)); origEdgeWidth[e.id] = e.width || 1; }});

  // 호버: 레이블
  network.on('hoverNode', function(p) {{
    var n = nodes.get(p.node);
    if (n && n.hiddenLabel) nodes.update({{id: p.node, label: n.hiddenLabel}});
  }});
  network.on('blurNode', function(p) {{ nodes.update({{id: p.node, label: ''}}); }});

  // 클릭: 1-hop
  var activeNode = null;
  network.on('click', function(params) {{
    if (params.nodes.length === 0) {{ restoreAll(); activeNode = null; return; }}
    var nid = params.nodes[0];
    if (activeNode === nid) {{ restoreAll(); activeNode = null; return; }}
    activeNode = nid;
    var neighbors = new Set(network.getConnectedNodes(nid));
    neighbors.add(nid);
    var connEdges = new Set(network.getConnectedEdges(nid));
    var nu = [];
    nodes.forEach(function(n) {{
      if (neighbors.has(n.id)) {{
        nu.push({{id: n.id, color: JSON.parse(JSON.stringify(origNodeColor[n.id])), opacity: 1.0, borderWidth: 3}});
      }} else {{
        nu.push({{id: n.id,
          color: {{background:'#e8ecef',border:'#ced4da',
                  highlight:{{background:'#e8ecef',border:'#ced4da'}},
                  hover:{{background:'#e8ecef',border:'#ced4da'}}}},
          opacity: 0.15}});
      }}
    }});
    nodes.update(nu);
    var eu = [];
    edges.forEach(function(e) {{
      if (connEdges.has(e.id)) {{
        eu.push({{id: e.id, color: JSON.parse(JSON.stringify(origEdgeColor[e.id])), width: origEdgeWidth[e.id] * 2}});
      }} else {{
        eu.push({{id: e.id, color:{{color:'#eeeeee'}}, width: 0.3}});
      }}
    }});
    edges.update(eu);
  }});

  function restoreAll() {{
    var nu = [];
    nodes.forEach(function(n) {{ nu.push({{id: n.id, color: JSON.parse(JSON.stringify(origNodeColor[n.id])), opacity: 1.0}}); }});
    nodes.update(nu);
    var eu = [];
    edges.forEach(function(e) {{ eu.push({{id: e.id, color: JSON.parse(JSON.stringify(origEdgeColor[e.id])), width: origEdgeWidth[e.id]}}); }});
    edges.update(eu);
  }}
</script>
</body>
</html>"""

html_path = os.path.join(OUT_DIR, 'compare_two_products.html')
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(html)
print(f'✅ HTML: {html_path}')
print(f'  노드 {len(vis_nodes)}개  엣지 {len(vis_edges)}개')
